# PySpark 11: structured streaming

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-11-structured-streaming.ipynb)

Orders, payments and clickstream events now arrive continuously. Finance still wants each event counted once,
product wants sessions as they happen, and the job must resume after a crash without losing or repeating data.

This lesson runs real Structured Streaming queries over local files: incremental processing with a checkpoint,
deduplication with a watermark, output modes and event-time windows, session windows, joins with static and
streaming data, progress metrics, and an idempotent `foreachBatch` merge into a Delta table. Kafka, custom state
and recovery close the lesson.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 7: aggregations and windows](pyspark-07-aggregations-and-windows.ipynb) and [PySpark 10: batch pipelines](pyspark-10-batch-pipelines.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 deltalake pyarrow

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#sample-data-used-in-every-module).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F
import json

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

In [ ]:
def land_json(directory, name, rows, mtime):
    """Write one JSON-lines file with a fixed modification time, so files are processed in order."""
    os.makedirs(directory, exist_ok=True)
    path = os.path.join(directory, name)
    with open(path, "w") as handle:
        handle.write("\n".join(json.dumps(row) for row in rows))
    os.utime(path, (mtime, mtime))
    return path


def run_until_done(query, timeout=90):
    """Wait for an availableNow query to finish, and always stop it."""
    try:
        if not query.awaitTermination(timeout):
            raise TimeoutError("stream did not finish")
    finally:
        query.stop()
    return query

## How a streaming query runs

A streaming DataFrame (`spark.readStream`) describes an unbounded table. Spark runs the same batch plan again on
each **trigger**, over the rows that arrived since the last one: a **micro-batch**. The **checkpoint** records
which input has been processed and the state of stateful operators, so a restarted query continues where it
stopped.

| piece | common choices |
| --- | --- |
| source | files (`json`, `parquet`, Auto Loader on Databricks), Kafka, `rate` for tests |
| sink | Delta or Iceberg tables (`toTable`), files, Kafka, `foreachBatch`, `memory` and `console` for tests |
| trigger | default (as fast as possible), `processingTime="1 minute"`, `availableNow=True` (process what is there, then stop) |

`trigger(availableNow=True)` is the scheduled-batch pattern with streaming bookkeeping: a job started every hour
processes only new files, and the checkpoint remembers which ones. `dropDuplicatesWithinWatermark` removes repeats
of a key across micro-batches, remembering keys only as long as the watermark allows.

**Where it matters**
- **Incremental file ingestion** without tracking processed files yourself.
- **Low-latency pipelines** from Kafka into lakehouse tables.

**Watch out for**
- One checkpoint per query, never shared, never moved. A new checkpoint path starts a new history and reprocesses everything.
- A sink can receive the same micro-batch again after a failure; it must be idempotent.

### Example

Land two files (the second repeats event `e2`), stream them one file per micro-batch, deduplicate within a
30-minute watermark, and write each batch to its own folder. Run the query again with the same checkpoint.

In [ ]:
demo_dir = tempfile.mkdtemp(prefix="stream-dedup-", dir=WORK_DIR)
landing_dir = os.path.join(demo_dir, "landing")
checkpoint_dir = os.path.join(demo_dir, "checkpoints", "events_silver")
silver_dir = os.path.join(demo_dir, "silver")
EVENT_SCHEMA = "event_id string, user_id int, event_ts timestamp, event_type string"

land_json(landing_dir, "part-0.json", [
    {"event_id": "e1", "user_id": 1, "event_ts": "2026-01-06T09:00:00", "event_type": "view"},
    {"event_id": "e2", "user_id": 1, "event_ts": "2026-01-06T09:02:00", "event_type": "add_to_cart"},
], mtime=1_000)
land_json(landing_dir, "part-1.json", [
    {"event_id": "e2", "user_id": 1, "event_ts": "2026-01-06T09:02:00", "event_type": "add_to_cart"},
    {"event_id": "e3", "user_id": 2, "event_ts": "2026-01-06T09:05:00", "event_type": "purchase"},
], mtime=2_000)

batch_ids = []


def write_batch(batch_df, batch_id):
    batch_ids.append(batch_id)
    # Idempotent per batch: a retried batch overwrites its own folder instead of appending twice.
    batch_df.write.mode("overwrite").parquet(os.path.join(silver_dir, f"batch_id={batch_id}"))


def run_dedup_stream():
    query = (
        spark.readStream.schema(EVENT_SCHEMA)
        .option("maxFilesPerTrigger", 1)
        .json(landing_dir)
        .withWatermark("event_ts", "30 minutes")
        .dropDuplicatesWithinWatermark(["event_id"])
        .writeStream.foreachBatch(write_batch)
        .option("checkpointLocation", checkpoint_dir)
        .trigger(availableNow=True)
        .start()
    )
    run_until_done(query)


def silver_event_ids():
    return sorted(r.event_id for r in spark.read.parquet(silver_dir).select("event_id").collect())

run_dedup_stream()
print("first run  -> batches:", batch_ids, "| silver events:", silver_event_ids())
run_dedup_stream()  # same checkpoint: both files are already committed
print("second run -> batches:", batch_ids, "| silver events:", silver_event_ids())

**What to notice:** The first run processed one file per micro-batch (batches 0 and 1), and `e2`, repeated in the second file, was
written once. Batch 2 is a no-data batch: Spark runs it to advance the watermark and clean up deduplication
state. The second run, on the same checkpoint, added no batch at all: the checkpoint already records both files
as processed.

## Choose an output mode and an event-time window

`F.window("event_ts", "5 minutes")` groups events into tumbling windows by **event time**, the time in the data,
not the arrival time; add a slide (`F.window(ts, "10 minutes", "5 minutes")`) for overlapping windows. A
**watermark** (`withWatermark("event_ts", "10 minutes")`) says how late data may arrive: once the latest event
time seen minus 10 minutes passes a window's end, the window is final and its state can be dropped.

The **output mode** decides what the sink receives on each trigger:

| mode | emits | needs |
| --- | --- | --- |
| `append` | only rows that will never change | a watermark for aggregations; rows appear after the window closes |
| `update` | rows that changed in this trigger | works with or without a watermark |
| `complete` | the whole result every time | aggregations only; state is never dropped |

**Where it matters**
- **Minute-by-minute dashboards:** `update` mode into a table keyed by window.
- **Final, immutable aggregates** written to files: `append` with a watermark.

**Watch out for**
- `complete` mode keeps every window forever; state grows without bound on a long-running stream.
- In `append` mode results appear only after the watermark passes the window end, so latency is at least the watermark delay.

### Example

Stream two files into five-minute windows. The second file adds an event to the first window and opens a new
one. Compare what the sink receives in `update` and `complete` mode, and read the progress metrics.

In [ ]:
window_dir = os.path.join(WORK_DIR, "stream-windows")
land_json(os.path.join(window_dir, "input"), "a.json",
          [{"ts": "2026-01-01 10:01:00", "amount": 10}, {"ts": "2026-01-01 10:02:00", "amount": 20}], mtime=1_000)
land_json(os.path.join(window_dir, "input"), "b.json",
          [{"ts": "2026-01-01 10:03:00", "amount": 5}, {"ts": "2026-01-01 10:07:00", "amount": 7}], mtime=2_000)


def run_windows(mode):
    totals = (
        spark.readStream.schema("ts timestamp, amount int")
        .option("maxFilesPerTrigger", 1)
        .json(os.path.join(window_dir, "input"))
        .withWatermark("ts", "10 minutes")
        .groupBy(F.window("ts", "5 minutes"))
        .agg(F.count("*").alias("events"), F.sum("amount").alias("amount"))
    )
    query = (
        totals.writeStream.format("memory").queryName(f"windows_{mode}").outputMode(mode)
        .option("checkpointLocation", os.path.join(window_dir, f"checkpoint-{mode}"))
        .trigger(availableNow=True).start()
    )
    run_until_done(query)
    rows = spark.table(f"windows_{mode}").collect()
    return sorted((r.window.start.strftime("%H:%M"), r.events) for r in rows), query

update_rows, update_query = run_windows("update")
complete_rows, _ = run_windows("complete")
print("update   (window start, events):", update_rows)
print("complete (window start, events):", complete_rows)

for progress in update_query.recentProgress:  # one entry per micro-batch
    p = progress if isinstance(progress, dict) else progress.jsonValue()
    print(f"batch {p['batchId']}: {p['numInputRows']} input rows, watermark {p['eventTime'].get('watermark')}")

**What to notice:** In `update` mode the sink received the 10:00 window twice: with 2 events after the first file and 3 after the
second, when the 10:03 event arrived. `complete` mode ends with the whole current result, one row per window.
The progress entries show the rows each micro-batch read and the watermark moving forward as later events
arrive: the numbers you watch on a production stream.

## Group activity into session windows

`F.session_window("ts", "10 minutes")` groups each key's events into sessions that close after 10 minutes without
activity: the window starts at the first event and ends 10 minutes after the last one. With a watermark and
`append` mode, a session is emitted once the watermark passes its end, so it is final when you see it.

**Where it matters**
- **Live web and app analytics:** sessions per user as they complete.
- **Device activity:** periods of use separated by idle time.

**Watch out for**
- A session that never goes quiet never closes; set a maximum or monitor state size.
- The batch version of the same idea uses `lag` and running sums ([lesson 7](pyspark-07-aggregations-and-windows.ipynb)).

### Example

User 1 is active at 10:00 and 10:05, then again at 10:40; user 2 once at 10:02. A last event at 12:00 moves the
watermark far enough for the earlier sessions to close and be emitted.

In [ ]:
session_dir = os.path.join(WORK_DIR, "stream-sessions")
land_json(os.path.join(session_dir, "input"), "a.json", [
    {"user_id": 1, "ts": "2026-01-06 10:00:00"}, {"user_id": 1, "ts": "2026-01-06 10:05:00"},
    {"user_id": 2, "ts": "2026-01-06 10:02:00"}, {"user_id": 1, "ts": "2026-01-06 10:40:00"},
], mtime=1_000)
land_json(os.path.join(session_dir, "input"), "b.json", [{"user_id": 9, "ts": "2026-01-06 12:00:00"}], mtime=2_000)

sessions = (
    spark.readStream.schema("user_id int, ts timestamp")
    .option("maxFilesPerTrigger", 1)
    .json(os.path.join(session_dir, "input"))
    .withWatermark("ts", "10 minutes")
    .groupBy("user_id", F.session_window("ts", "10 minutes"))
    .agg(F.count("*").alias("events"))
)
query = (
    sessions.writeStream.format("memory").queryName("sessions").outputMode("append")
    .option("checkpointLocation", os.path.join(session_dir, "checkpoint"))
    .trigger(availableNow=True).start()
)
run_until_done(query)
spark.table("sessions").select(
    "user_id", F.col("session_window.start").alias("start"), F.col("session_window.end").alias("end"), "events"
).orderBy("user_id", "start").show()

**What to notice:** User 1 has two sessions: 10:00 to 10:15 (two events; the session ends 10 minutes after the 10:05 event) and a
separate one from 10:40, because 35 minutes passed. User 2 has one. User 9's session is not in the output yet:
in `append` mode a session is emitted only after the watermark passes its end, and nothing later has arrived to
move it.

## Join streams with static and streaming data

- A **stream-static join** enriches each micro-batch with a batch DataFrame, such as a customer dimension. Spark
  reads the static side again on each trigger, so a dimension table that changes is picked up.
- A **stream-stream join** matches rows from two streams, for example orders with their payments. Spark buffers
  rows from both sides in state until they can no longer match. That requires a **watermark on both sides** and
  a **time-range condition** in the join, so Spark knows when to drop buffered rows.

**Where it matters**
- **Enriching events** with customer region or product category before writing silver.
- **Matching related events:** an order and its payment within 30 minutes.

**Watch out for**
- A large static side is re-read on every trigger; keep it small or broadcast it.
- A stream-stream join without watermarks and a time bound keeps every row in state forever.

### Example

Enrich streamed events with the customer region. User 9 has no customer row and is kept by the left join.

In [ ]:
join_dir = os.path.join(WORK_DIR, "stream-static")
land_json(os.path.join(join_dir, "input"), "events.json", [
    {"event_id": "j1", "user_id": 1, "event_ts": "2026-01-06T09:00:00", "event_type": "view"},
    {"event_id": "j2", "user_id": 3, "event_ts": "2026-01-06T09:01:00", "event_type": "view"},
    {"event_id": "j3", "user_id": 9, "event_ts": "2026-01-06T09:02:00", "event_type": "view"},
], mtime=1_000)

regions = customers.select(F.col("customer_id").alias("user_id"), "region")
enriched = spark.readStream.schema(EVENT_SCHEMA).json(os.path.join(join_dir, "input")).join(regions, "user_id", "left")
query = (
    enriched.writeStream.format("memory").queryName("enriched_events").outputMode("append")
    .option("checkpointLocation", os.path.join(join_dir, "checkpoint"))
    .trigger(availableNow=True).start()
)
run_until_done(query)
spark.table("enriched_events").orderBy("event_id").show()

**What to notice:** Each streamed event picked up its customer's region. User 9 has no customer row; the `left` join kept the event
with a null region instead of dropping it.

### Example: a stream-stream join

Match an order stream with a payment stream: a payment counts only if it arrives within 30 minutes of its order.
Order 3's payment comes too late and order 2 is never paid.

In [ ]:
orders_dir, payments_dir = os.path.join(WORK_DIR, "s_orders"), os.path.join(WORK_DIR, "s_payments")
land_json(orders_dir, "o.json", [
    {"order_id": 1, "order_ts": "2026-01-06 10:00:00"}, {"order_id": 2, "order_ts": "2026-01-06 10:05:00"},
    {"order_id": 3, "order_ts": "2026-01-06 10:10:00"},
], mtime=1_000)
land_json(payments_dir, "p.json", [
    {"pay_order_id": 1, "paid_ts": "2026-01-06 10:12:00"}, {"pay_order_id": 3, "paid_ts": "2026-01-06 11:30:00"},
], mtime=1_000)

order_stream = spark.readStream.schema("order_id int, order_ts timestamp").json(orders_dir).withWatermark("order_ts", "1 hour")
payment_stream = spark.readStream.schema("pay_order_id int, paid_ts timestamp").json(payments_dir).withWatermark("paid_ts", "1 hour")

paid = order_stream.join(
    payment_stream,
    F.expr("pay_order_id = order_id AND paid_ts BETWEEN order_ts AND order_ts + INTERVAL 30 MINUTES"),
)
query = (
    paid.writeStream.format("memory").queryName("paid_orders").outputMode("append")
    .option("checkpointLocation", os.path.join(WORK_DIR, "s_join_checkpoint"))
    .trigger(availableNow=True).start()
)
run_until_done(query)
spark.table("paid_orders").orderBy("order_id").show()

**What to notice:** Only order 1 matched: its payment came 12 minutes after the order. Order 3's payment arrived 80 minutes later,
outside the 30-minute condition, and order 2 has no payment. The time bound and the two watermarks let Spark drop
buffered orders once no payment can match them any more.

## Write each micro-batch with foreachBatch

`foreachBatch(fn)` calls `fn(batch_df, batch_id)` with each micro-batch as an ordinary DataFrame, so any batch
write can be the sink: a `MERGE` into Delta or Iceberg, writes to several tables, or a JDBC upsert. Because a
batch can be delivered again after a failure, the write must be idempotent: merge by key and source version, or
overwrite a location the batch owns.

**Where it matters**
- **Upserting a change stream** into a silver table.
- **Fan-out:** one stream written to a table and to an audit log.

**Watch out for**
- Never `collect()` the batch to the driver; keep it distributed.
- Cache the batch if `fn` writes it more than once, and unpersist it at the end.

### Example

Stream order changes into a Delta table with a version-aware merge in `foreachBatch`. The second file corrects
order 1 and repeats order 2's old version. Then deliver the last batch again, as Spark would after a crash.

In [ ]:
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake

delta_path = os.path.join(WORK_DIR, "delta_orders_stream")
write_deltalake(delta_path, pa.table({"order_id": pa.array([], pa.int32()), "status": pa.array([], pa.string()),
                                      "version": pa.array([], pa.int64())}))
changes_dir = os.path.join(WORK_DIR, "order_changes")
land_json(changes_dir, "1.json", [{"order_id": 1, "status": "placed", "version": 1},
                                  {"order_id": 2, "status": "placed", "version": 1}], mtime=1_000)
land_json(changes_dir, "2.json", [{"order_id": 1, "status": "paid", "version": 2},
                                  {"order_id": 2, "status": "placed", "version": 1}], mtime=2_000)


def merge_into_delta(batch_df, batch_id):
    latest = batch_df.groupBy("order_id").agg(F.max_by(F.struct("status", "version"), "version").alias("c")) \
        .select("order_id", "c.status", "c.version")
    (
        DeltaTable(delta_path)
        .merge(latest.toArrow(), predicate="t.order_id = s.order_id", source_alias="s", target_alias="t")
        .when_matched_update_all(predicate="s.version > t.version")
        .when_not_matched_insert_all()
        .execute()
    )
    print(f"batch {batch_id}: merged {latest.count()} changes")

query = (
    spark.readStream.schema("order_id int, status string, version long").option("maxFilesPerTrigger", 1)
    .json(changes_dir).writeStream.foreachBatch(merge_into_delta)
    .option("checkpointLocation", os.path.join(WORK_DIR, "delta_stream_checkpoint"))
    .trigger(availableNow=True).start()
)
run_until_done(query)
replay = spark.read.schema("order_id int, status string, version long").json(os.path.join(changes_dir, "2.json"))
merge_into_delta(replay, 1)  # the same batch delivered again after a restart

print(sorted(DeltaTable(delta_path).to_pyarrow_table().to_pylist(), key=lambda r: r["order_id"]))
print("table versions:", [entry["operation"] for entry in reversed(DeltaTable(delta_path).history())])

**What to notice:** Order 1 ended `paid` at version 2; order 2 stayed `placed`, version 1. The repeated version-1 row for order 2 and
the replayed batch matched existing rows but failed `s.version > t.version`, so they changed nothing, and the
history shows no extra commits for them. `DataFrame.toArrow()` (Spark 4) handed each batch to the Delta library
without going through pandas.

## Read from Kafka

The Kafka source reads topics as rows of `key`, `value` (binary), `topic`, `partition`, `offset` and `timestamp`.
Decode `value` with `from_json` and a schema, keep topic, partition and offset for lineage, and bound each
micro-batch with `maxOffsetsPerTrigger`. The checkpoint stores the offsets, so `startingOffsets` applies only to
a brand-new query.

```python
orders_stream = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("subscribe", "orders")
    .option("startingOffsets", "earliest")      # first run only; the checkpoint wins afterwards
    .option("maxOffsetsPerTrigger", 500_000)    # bound each micro-batch
    .load()
    .select(F.from_json(F.col("value").cast("string"), order_schema).alias("o"), "topic", "partition", "offset")
    .select("o.*", "topic", "partition", "offset")
)
(
    orders_stream.writeStream
    .foreachBatch(merge_into_silver)             # idempotent MERGE per batch
    .option("checkpointLocation", "s3://example-checkpoints/orders/")
    .trigger(processingTime="1 minute")
    .start()
)
```

The Kafka connector is a separate package (`spark-sql-kafka-0-10`) and needs a broker, so this is shown, not run.

**Where it matters**
- **Event pipelines** from application services into lakehouse tables.
- **CDC** from Debezium topics into silver tables with a merge per batch.

**Watch out for**
- Without `maxOffsetsPerTrigger`, a backlog becomes one enormous first batch.
- Malformed messages make `from_json` return null; quarantine them with topic, partition and offset.
- Kafka retention shorter than your longest outage means the offsets in the checkpoint may no longer exist.

### Example

Size a trigger: a topic receives 2,000 events a second and each micro-batch should cover at most a minute of
traffic, so `maxOffsetsPerTrigger = 2,000 x 60 = 120,000`, spread over the topic's partitions. If a batch takes
longer than the trigger interval, batches start late and latency climbs; add resources or widen the interval.

## Keep custom per-key state

Built-in stateful operators (windowed aggregation, `dropDuplicatesWithinWatermark`, stream-stream joins) cover most
pipelines. When the rule is your own ("alert on the third failed payment in an hour"), Spark 4 adds
`transformWithStateInPandas`. You write a `StatefulProcessor`: `init` declares named state variables (`ValueState`,
`ListState`, `MapState`), each with an optional TTL; `handleInputRows` runs once per key per micro-batch; timers
fire `handleExpiredTimer`.

```python
class FailedPaymentAlert(StatefulProcessor):
    def init(self, handle):
        self.failures = handle.getValueState("failures", "n long", ttlDurationMs=3_600_000)  # expires after an hour

    def handleInputRows(self, key, rows, timerValues):
        total = (self.failures.get()[0] if self.failures.exists() else 0) + sum(len(batch) for batch in rows)
        self.failures.update((total,))
        if total >= 3:
            yield pd.DataFrame({"account_id": [key[0]], "failures": [total]})

    def close(self):
        pass

alerts = failed_payments.groupBy("account_id").transformWithStateInPandas(
    statefulProcessor=FailedPaymentAlert(), outputStructType="account_id string, failures long",
    outputMode="Append", timeMode="None",
)
```

It needs the RocksDB state store (`spark.sql.streaming.stateStore.providerClass`) and `pandas`, `pyarrow` and
`protobuf` on the workers, so it is shown, not run.

**Where it matters**
- **Fraud and alerting rules** that count events per account over time.
- **Custom sessionization** with business-specific timeouts.

**Watch out for**
- State with no TTL or timer: a key that never returns keeps its state forever.
- Changing the state schema can make an existing checkpoint unusable; plan it like a table migration.

### Example

Estimate state before shipping: 5 million active accounts with 200 bytes of state each is about 1 GB, checkpointed
on every trigger. A one-hour TTL keeps it to accounts that failed recently instead of every account ever seen.

## Recover a query after a failure

On restart with the same checkpoint, Spark replays any micro-batch that was not recorded as committed, so your
sink can receive the same `batch_id` and rows twice. Make that harmless: overwrite a location owned by the batch,
merge by key and version (as above), or store the batch id in the same transaction as the output.

**Where it matters**
- **Driver crashes and cluster restarts** between the sink write and the checkpoint commit.
- **Deployments:** new code restarting on an old checkpoint.

**Watch out for**
- Changing the source, the stateful operators or the state schema may make the checkpoint incompatible.
- A `batch_id` is only unique within one checkpoint's history.

### Example

Deliver the same batch to the first example's sink twice, as Spark would after a crash.

In [ ]:
replayed = spark.createDataFrame(
    [("e9", 3, datetime(2026, 1, 6, 9, 30), "view"), ("e10", 3, datetime(2026, 1, 6, 9, 31), "view")], EVENT_SCHEMA
)
write_batch(replayed, 99)
write_batch(replayed, 99)  # the same batch again after a restart
spark.read.parquet(os.path.join(silver_dir, "batch_id=99")).orderBy("event_id").show()

**What to notice:** Batch 99 was written twice, but its folder holds each event once: the second write replaced the first. A plain
append in `foreachBatch` would have stored four rows.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** What does the checkpoint store, and what happens if you change its path?

**A.** Processed input (files or offsets), committed batches and operator state. A new path starts a new history: all input is processed again and state is lost.

**Q2.** A watermark is 10 minutes and the latest event time seen is 10:40. Which windows are final?

**A.** Windows ending at or before 10:30, the watermark. Their state can be dropped and, in append mode, emitted.

**Q3.** When do you use `update` instead of `complete` mode?

**A.** For aggregations on long-running streams: `update` emits only changed rows and lets the watermark drop old state; `complete` re-emits and keeps everything.

**Q4.** What does a stream-stream join need, and why?

**A.** Watermarks on both sides and a time-range condition, so Spark can drop buffered rows that can no longer match.

**Q5.** Why must a `foreachBatch` function be idempotent?

**A.** After a failure Spark can call it again with the same batch id and rows; a plain append would write them twice.

**Q6.** When is a session emitted in append mode?

**A.** When the watermark passes the session's end, that is, after the inactivity gap plus the allowed lateness.

## Key takeaways

- A streaming query is a batch plan re-run per trigger; the checkpoint makes it restartable. One per query.
- Watermarks bound lateness and state; without them, state grows forever.
- Use `update` or `append` for long-running aggregations; `complete` keeps everything.
- Stream-stream joins need watermarks on both sides and a time bound.
- Make sinks idempotent: overwrite what a batch owns, or merge by key and version.

**Next:** Practise this module in [PySpark drills 11: structured streaming](pyspark-drills-11-structured-streaming.ipynb), then continue with [PySpark 12: internals and monitoring](pyspark-12-internals-and-monitoring.ipynb).